# Grade 11 · Unit 2 — SQL windows and joins

> **SYNTHETIC** district and schools tables.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def data_file(dataset, filename):
    """Find datasets/<dataset>/clean/<filename> by walking up from this folder."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        p = folder / "datasets" / dataset / "clean" / filename
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {filename}. Open this notebook from inside the data-science-nepal folder.")
import sqlite3
districts = pd.read_csv(data_file("census-districts", "districts_synthetic.csv"))
schools = pd.read_csv(data_file("census-districts", "schools_synthetic.csv"))
con = sqlite3.connect(":memory:")
districts.to_sql("districts", con, index=False); schools.to_sql("schools", con, index=False)
q = lambda sql: pd.read_sql_query(sql, con)
print(sqlite3.sqlite_version)

## RANK within a group
**Predict:** what is the highest rank in each belt?

In [ ]:
ranked = q("""
SELECT district, ecological_belt, population,
       RANK() OVER (PARTITION BY ecological_belt ORDER BY population DESC) AS rank_in_belt
FROM districts
""")
sizes = districts["ecological_belt"].value_counts()
assert (ranked.groupby("ecological_belt")["rank_in_belt"].max() == sizes.sort_index()).all()
ranked[ranked["rank_in_belt"] <= 3].sort_values(["ecological_belt", "rank_in_belt"])

## Share of the belt, and a running total

In [ ]:
q("""
SELECT district, ecological_belt, population,
       ROUND(100.0 * population / SUM(population) OVER (PARTITION BY ecological_belt), 1) AS pct_of_belt,
       SUM(population) OVER (PARTITION BY ecological_belt ORDER BY population DESC) AS running_total
FROM districts
ORDER BY ecological_belt, population DESC
""").head(8)

## INNER vs LEFT JOIN
We delete three school rows to see the difference.

In [ ]:
con.execute("DELETE FROM schools WHERE district_id IN (1, 2, 3)")
inner = q("SELECT COUNT(*) AS n FROM districts d JOIN schools s USING (district_id)")["n"][0]
left = q("SELECT COUNT(*) AS n FROM districts d LEFT JOIN schools s USING (district_id)")["n"][0]
assert (inner, left) == (74, 77)
print("INNER:", inner, "rows | LEFT:", left, "rows (unmatched districts kept, with NULLs)")

**Make:** find the top-2 districts per province by population.